# 🐝 Lab 4.3 — Multimodal Magic: Contrastive Learning & CLIP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab12_clip_multimodal.ipynb)

⏱ **Time:** ~1.5 hours  |  📅 **Day 4 — Generative Worlds**  |  Runs on CPU; a GPU just makes the pretrained-CLIP part faster.

🎯 **Learning objectives** — by the end of this lab you will be able to:
- Derive and implement the **symmetric InfoNCE (CLIP) loss** with a learnable temperature, and explain what it optimizes.
- Train a tiny **two-tower** image/text model from scratch and watch the similarity matrix become **diagonal**.
- Use a pretrained **CLIP** for **zero-shot classification** with prompt engineering and **prompt ensembling**, and for image↔text **retrieval** (with Recall@K).
- Visualize a joint embedding space with PCA / t-SNE and measure the **modality gap**.
- Implement the **SigLIP** sigmoid loss and compare it to softmax InfoNCE.
- Explain how modern **vision-language models** plug a vision encoder into an LLM.

🧰 **Prerequisites:** PyTorch basics, cross-entropy/softmax, cosine similarity. Lab 4.2 is helpful but not required.

**Why this matters in 2026.** Contrastive image–text encoders are the quiet workhorses of multimodal AI: they rank and filter web-scale training data, power semantic image search, provide the text conditioning or evaluation metric (CLIP-score) for image generators, and — most importantly — serve as the **eyes of vision-language models** (SigLIP-family encoders sit inside many open VLMs). Everything rests on one elegant loss you'll write in about six lines. 🍯

### 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| Setup | Imports, seeds, palette | 5 |
| 1 | Derive and implement the symmetric InfoNCE / CLIP loss | 15 |
| 2 | Train a tiny CLIP from scratch on colored MNIST ↔ captions and **watch the similarity matrix snap to the diagonal** (animated) | 20 |
| 3 | Pretrained CLIP: zero-shot CIFAR-10, prompt ensembling, retrieval, Recall@K, embedding-space maps | 25 |
| 4 | Implement SigLIP's sigmoid loss and compare | 10 |
| 5 | How VLMs plug a vision encoder into an LLM | 10 |
| Wrap-up | Boss Level, key takeaways, interview corner | 5 |

In [ ]:
# 🛠️ Setup — torch, torchvision, transformers, sklearn and matplotlib are preinstalled on Colab
import os, math, time, random, re
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (tiny runs). Normal users: leave it off.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = os.environ.get("LAB_DATA", "./data")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # silence a harmless HF tokenizers fork warning
import contextlib, io, warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

@contextlib.contextmanager
def quiet():
    """Swallow download progress bars / chatter (errors still raise)."""
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        yield

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(42)

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA", "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})

print(f"torch {torch.__version__} | device = {DEVICE} | FAST = {FAST}")

## Part 1 — The contrastive objective: InfoNCE / CLIP loss 🧲

**Intuition.** We have $N$ matching (image, caption) pairs in a batch. Encode images with $f$ and texts with $g$, L2-normalize, and compute all $N^2$ cosine similarities. The loss asks: *for each image, pick its own caption out of the $N$ candidates* — and vice versa. That's just **$N$-way classification**, where the "classes" are the other items in the batch. Positives get pulled together, the $N-1$ in-batch negatives get pushed apart.

**Math.** With normalized embeddings $u_i = f(x_i)/\|f(x_i)\|$, $v_j = g(y_j)/\|g(y_j)\|$ and a learnable temperature $\tau$ (CLIP stores $s = \log(1/\tau)$, initialized to $\log(1/0.07)$, and clips $1/\tau \le 100$):
$$\ell_{ij} = \frac{u_i^\top v_j}{\tau},\qquad
\mathcal{L}_{\text{CLIP}} = \frac{1}{2N}\sum_{i=1}^N\Big[\underbrace{-\log\frac{e^{\ell_{ii}}}{\sum_j e^{\ell_{ij}}}}_{\text{image}\to\text{text}} \;\underbrace{-\log\frac{e^{\ell_{ii}}}{\sum_j e^{\ell_{ji}}}}_{\text{text}\to\text{image}}\Big].$$
In code: `cross_entropy(logits, arange(N))` on the rows and on the columns, averaged.

**Why it works.** InfoNCE (van den Oord et al., 2018) is a lower bound on mutual information, $I(X;Y) \ge \log N - \mathcal{L}_{\text{InfoNCE}}$: bigger batches mean more negatives and a tighter bound — one reason CLIP used a batch of 32,768. A random model has $\mathcal{L}\approx \log N$. The temperature controls how sharply the softmax focuses on hard negatives.

### 🧪 Your Turn #1 — implement the CLIP loss

`clip_loss(img_emb, txt_emb, logit_scale)`: normalize both embeddings, form the $N\times N$ logit matrix scaled by `logit_scale` (already $=1/\tau$, a positive number or 0-dim tensor), and return the symmetric cross-entropy.

In [ ]:
def clip_loss(img_emb, txt_emb, logit_scale):
    # TODO 1: L2-normalize both (F.normalize(..., dim=-1))
    # TODO 2: logits = logit_scale * img @ txt.T          (shape N x N)
    # TODO 3: targets = torch.arange(N)
    # TODO 4: return average of cross-entropy over rows (image->text) and columns (text->image)
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def clip_loss(img_emb, txt_emb, logit_scale):
    img, txt = F.normalize(img_emb, dim=-1), F.normalize(txt_emb, dim=-1)
    logits = logit_scale * img @ txt.T
    targets = torch.arange(len(img), device=img.device)
    return 0.5 * (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets))

In [ ]:
# ✔️ Checkpoint — compare against a slow, explicit double loop
def _reference_clip_loss(a, b, s):
    a = [x / x.norm() for x in a]; b = [y / y.norm() for y in b]; N = len(a); total = 0.0
    for i in range(N):
        row = torch.stack([s * (a[i] @ b[j]) for j in range(N)])
        col = torch.stack([s * (a[j] @ b[i]) for j in range(N)])
        total += -(row[i] - torch.logsumexp(row, 0)) - (col[i] - torch.logsumexp(col, 0))
    return total / (2 * N)

torch.manual_seed(0)
_a, _b = torch.randn(16, 8), torch.randn(16, 8)
assert torch.allclose(clip_loss(_a, _b, 10.0), _reference_clip_loss(_a, _b, 10.0), atol=1e-5), "doesn't match the reference"
assert abs(clip_loss(_a, _b, 0.0).item() - math.log(16)) < 1e-5, "with logit_scale=0 every candidate is equally likely -> log N"
_e = torch.eye(16)
assert clip_loss(_e, _e, 100.0).item() < 1e-6, "perfectly aligned orthogonal embeddings -> loss ~ 0"
print(f"✔️ clip_loss correct.  Random embeddings, N=16: loss {clip_loss(_a, _b, 1.0).item():.3f} at 1/τ=1 (≈ log N = {math.log(16):.3f}), "
      f"but {clip_loss(_a, _b, 1 / 0.07).item():.3f} at 1/τ=1/0.07 — a sharp softmax punishes confident mistakes!")

> **🍯 Key takeaway:** CLIP's loss is $N$-way classification inside every batch, run in both directions (image→text and text→image). The temperature controls how hard it focuses on confusable negatives.

## Part 2 — A tiny CLIP from scratch: Colored MNIST ↔ captions 🎨🔢

We need **paired** data. We'll make our own: tint MNIST digits **red, green or blue** and pair each image with a randomly-templated caption like *"a red handwritten seven"*, *"the digit 7 in green ink"*, or *"blue 3"*. That gives $10 \times 3 = 30$ visual concepts and requires the model to learn that "seven" and "7" mean the same thing — only from co-occurrence!

**Towers.**
- *Image tower:* a 3-layer CNN → 64-d embedding.
- *Text tower:* word embeddings, **mean-pooled** (a bag of words), then a 2-layer MLP → 64-d. Deliberately tiny — word order doesn't matter in our captions. Real CLIP uses a Transformer.

In [ ]:
from torchvision import datasets
try:
    with quiet():                                   # hide download progress bars
        _tr = datasets.MNIST(DATA_ROOT, train=True, download=True)
        _te = datasets.MNIST(DATA_ROOT, train=False, download=True)
    Xtr, Ytr = _tr.data.float().div(255), _tr.targets.long()
    Xte, Yte = _te.data.float().div(255), _te.targets.long()
except Exception as e:
    print(f"⚠️ MNIST download failed ({type(e).__name__}); using sklearn's 8x8 digits upsampled to 28x28")
    from sklearn.datasets import load_digits
    d = load_digits()
    Xall = F.interpolate(torch.tensor(d.images).float().div(16)[:, None], size=28, mode="bilinear")[:, 0]
    Yall = torch.tensor(d.target).long()
    Xtr, Ytr, Xte, Yte = Xall[:1500], Yall[:1500], Xall[1500:], Yall[1500:]
if FAST:
    Xtr, Ytr, Xte, Yte = Xtr[:3000], Ytr[:3000], Xte[:600], Yte[:600]

COLOR_NAMES = ["red", "green", "blue"]
COLOR_RGB = torch.tensor([[1.0, 0.25, 0.2], [0.2, 0.85, 0.3], [0.25, 0.45, 1.0]])
DIGIT_WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
TEMPLATES = ["a {c} handwritten {w}", "the digit {n} in {c} ink", "a {c} number {w}",
             "{c} {n}", "a sloppy {c} {w}", "the number {w} written in {c}"]

def caption(d, c, k):
    return TEMPLATES[k].format(c=COLOR_NAMES[c], w=DIGIT_WORDS[d], n=str(d))

def colorize(gray, color_idx):
    """gray: (B, 28, 28) in [0,1]  ->  (B, 3, 28, 28) in [-1, 1]"""
    return (gray[:, None] * COLOR_RGB[color_idx][:, :, None, None]) * 2 - 1

# --- a minimal word-level tokenizer -------------------------------------------------
def words(s): return re.findall(r"[a-z]+|\d", s.lower())
VOCAB = ["<pad>", "<unk>"] + sorted({w for d in range(10) for c in range(3) for k in range(len(TEMPLATES)) for w in words(caption(d, c, k))})
W2I = {w: i for i, w in enumerate(VOCAB)}
MAX_LEN = 10
def tokenize(texts):
    out = torch.zeros(len(texts), MAX_LEN, dtype=torch.long)
    for i, s in enumerate(texts):
        ids = [W2I.get(w, 1) for w in words(s)][:MAX_LEN]
        out[i, :len(ids)] = torch.tensor(ids)
    return out
# Pre-tokenize every (digit, color, template) caption once: (10, 3, K, MAX_LEN)
TOK = torch.stack([torch.stack([tokenize([caption(d, c, k) for k in range(len(TEMPLATES))]) for c in range(3)]) for d in range(10)])
print(f"vocab size {len(VOCAB)}: {VOCAB}")

def get_batch(X, Y, n, gen=None):
    idx = torch.randint(0, len(X), (n,), generator=gen)
    col = torch.randint(0, 3, (n,), generator=gen)
    tmpl = torch.randint(0, len(TEMPLATES), (n,), generator=gen)
    return colorize(X[idx], col), TOK[Y[idx], col, tmpl], Y[idx], col, tmpl

_g = torch.Generator().manual_seed(0)
_img, _tok, _y, _c, _k = get_batch(Xtr, Ytr, 8, _g)
fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
for i, ax in enumerate(axes):
    ax.imshow(((_img[i].permute(1, 2, 0) + 1) / 2).clamp(0, 1)); ax.axis("off")
    ax.set_title(caption(_y[i].item(), _c[i].item(), _k[i].item()), fontsize=8.5)
plt.suptitle("Paired training data: image ↔ caption", y=1.05); plt.show()

In [ ]:
EMB = 64

class ImageTower(nn.Module):
    def __init__(self, d=EMB):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(),     # 14x14
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),    # 7x7
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),   # 4x4
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, d))
    def forward(self, x): return self.net(x)

class TextTower(nn.Module):
    def __init__(self, vocab_size, d=EMB, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden, padding_idx=0)
        self.mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, d))
    def forward(self, tok):
        mask = (tok != 0).float()[..., None]
        pooled = (self.emb(tok) * mask).sum(1) / mask.sum(1).clamp(min=1)   # mean over real tokens
        return self.mlp(pooled)

class TinyCLIP(nn.Module):
    def __init__(self, vocab_size, init_scale=1 / 0.07, init_bias=None):
        super().__init__()
        self.image_tower, self.text_tower = ImageTower(), TextTower(vocab_size)
        self.log_scale = nn.Parameter(torch.tensor(math.log(init_scale)))
        self.bias = nn.Parameter(torch.tensor(float(init_bias))) if init_bias is not None else None   # used by SigLIP
    def encode_image(self, x): return F.normalize(self.image_tower(x), dim=-1)
    def encode_text(self, tok): return F.normalize(self.text_tower(tok), dim=-1)
    def scale(self): return self.log_scale.clamp(max=math.log(100)).exp()      # CLIP caps 1/tau at 100

# A "showcase" batch: one image per digit, cycling through colors, with its canonical caption
_gs = torch.Generator().manual_seed(1)
show_idx = torch.stack([torch.nonzero(Yte == d)[torch.randint(0, int((Yte == d).sum()), (1,), generator=_gs)].squeeze() for d in range(10)])
show_col = torch.arange(10) % 3
show_img = colorize(Xte[show_idx], show_col)
show_txt = [caption(d, show_col[d].item(), 0) for d in range(10)]
show_tok = tokenize(show_txt)

def plot_sim(model, title, ax):
    model.eval()
    with torch.no_grad():
        S = (model.encode_image(show_img.to(DEVICE)) @ model.encode_text(show_tok.to(DEVICE)).T).cpu()
    ax.imshow(S, cmap="magma", vmin=-1, vmax=1)
    ax.set_xticks(range(10)); ax.set_xticklabels(show_txt, rotation=60, ha="right", fontsize=7.5)
    ax.set_yticks(range(10)); ax.set_yticklabels([f"img: {t.split()[1]} {DIGIT_WORDS[i]}" for i, t in enumerate(show_txt)], fontsize=7.5)
    ax.set_title(title)
    return S

seed_everything(0)
toy_clip = TinyCLIP(len(VOCAB)).to(DEVICE)
fig, ax = plt.subplots(figsize=(5, 5)); S_before = plot_sim(toy_clip, "Cosine similarity BEFORE training", ax); plt.show()

### 🔮 Predict-then-run

We'll train with batch size 256. With only 30 distinct concepts, a batch of 256 contains *many* pairs sharing the same concept (e.g. ~8 "red seven"s). The CLIP loss treats all of them as **negatives** for each other.
- Will the model still learn? Will the loss go to zero?
- What happens to the off-diagonal entries of the showcase similarity matrix between, say, "red seven" and "green seven"?

In [ ]:
def train_contrastive(model, loss_kind="clip", steps=None, batch=256, lr=2e-3, seed=0, n_snapshots=0):
    """n_snapshots > 0 also records the showcase similarity matrix during training (for the animation)."""
    steps = steps or (60 if FAST else 1500)
    snap_every = max(1, steps // n_snapshots) if n_snapshots else None
    seed_everything(seed); gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    model.train(); hist = {"loss": [], "scale": [], "snaps": []}; t0 = time.time()
    for step in range(steps + 1):
        if snap_every and step % snap_every == 0:
            model.eval()
            with torch.no_grad():
                hist["snaps"].append((step, (model.encode_image(show_img.to(DEVICE)) @ model.encode_text(show_tok.to(DEVICE)).T).cpu()))
            model.train()
        if step == steps:
            break
        img, tok, *_ = get_batch(Xtr, Ytr, batch, gen)
        u = model.image_tower(img.to(DEVICE)); v = model.text_tower(tok.to(DEVICE))
        if loss_kind == "clip":
            loss = clip_loss(u, v, model.scale())
        else:
            loss = siglip_loss(u, v, model.log_scale.exp(), model.bias)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        hist["loss"].append(loss.item()); hist["scale"].append(model.log_scale.exp().item())
    print(f"[{loss_kind}] {steps} steps in {time.time() - t0:.0f}s | final loss {np.mean(hist['loss'][-50:]):.3f} | 1/tau = {hist['scale'][-1]:.1f}")
    return model.eval(), hist

toy_clip, hist_clip = train_contrastive(toy_clip, "clip", n_snapshots=30)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].plot(hist_clip["loss"], lw=0.8); axes[0].axhline(math.log(256), ls="--", c="gray", label="log N (chance)")
axes[0].set_xlabel("step"); axes[0].set_ylabel("CLIP loss"); axes[0].legend(); axes[0].set_title("Loss")
ax2 = axes[0].twinx(); ax2.plot(hist_clip["scale"], c=HIVE[2], lw=1); ax2.set_ylabel("learned 1/τ", color=HIVE[2]); ax2.spines["right"].set_visible(True)
plot_sim(TinyCLIP(len(VOCAB)).to(DEVICE), "BEFORE (random init)", axes[1])
S_after = plot_sim(toy_clip, "AFTER training", axes[2])
plt.tight_layout(); plt.show()

### 🎬 The wow moment: alignment emerging

The same 10×10 showcase matrix, recorded throughout training. Press ▶ and watch the image and text towers, which start out knowing nothing about each other, agree on a shared language: the diagonal lights up.

In [ ]:
from matplotlib import animation
from IPython.display import HTML
plt.rcParams["animation.embed_limit"] = 30          # MB

snaps = hist_clip["snaps"] + [hist_clip["snaps"][-1]] * 4     # linger on the final frame
fig, ax = plt.subplots(figsize=(5.2, 5.0), dpi=80)
im = ax.imshow(snaps[0][1], cmap="magma", vmin=-1, vmax=1)
ax.set_xticks(range(10)); ax.set_xticklabels([t.split()[1] + " " + t.split()[-1] for t in show_txt], rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(10)); ax.set_yticklabels([f"img {d}" for d in range(10)], fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, label="cosine similarity")
ttl = ax.set_title("")
diag = lambda S: S.diag().mean().item() - (S.sum() - S.diag().sum()).item() / 90
def update(k):
    step, S = snaps[k]
    im.set_data(S); ttl.set_text(f"step {step}: diag − off-diag = {diag(S):.2f}")
    return im, ttl
anim = animation.FuncAnimation(fig, update, frames=len(snaps), interval=150)
plt.tight_layout(); plt.close(fig)
HTML(anim.to_jshtml())

**Reveal.** The similarity matrix turns sharply **diagonal**, but the loss plateaus well above 0: with ~8 identical concepts per batch, the model *cannot* tell which of the 8 "red seven" captions belongs to which red seven image — these **false negatives** put a floor on the loss (roughly the log of the number of duplicates). Real web data has the same problem in milder form (many near-duplicate captions). Fixes include larger/more diverse data, soft or label-aware targets (see Boss Level), and the SigLIP loss in Part 4.

### Zero-shot classification with our tiny CLIP

"Classify" a test image by embedding a text prompt for every concept and picking the most similar one — **no classifier head is ever trained**. Averaging embeddings over several prompt templates (**prompt ensembling**) usually helps.

### 🧪 Your Turn #2 — build zero-shot classifier weights

`build_zeroshot_weights(encode_text_fn, classnames, templates)`:
for each class, format every template with the class name, encode the batch of prompts (`encode_text_fn(list_of_str)` returns **normalized** `(T, D)` embeddings), **average** them, **re-normalize**, and stack into a `(C, D)` matrix. This one function works for our toy model *and* for real CLIP.

In [ ]:
def build_zeroshot_weights(encode_text_fn, classnames, templates):
    # TODO: for each class -> texts = [t.format(name) for t in templates]; e = encode_text_fn(texts)
    #       mean over templates, re-normalize; return torch.stack(...)  shape (C, D)
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
@torch.no_grad()
def build_zeroshot_weights(encode_text_fn, classnames, templates):
    W = []
    for name in classnames:
        e = encode_text_fn([t.format(name) for t in templates])
        W.append(F.normalize(e.mean(0), dim=-1))
    return torch.stack(W)

In [ ]:
# ✔️ Checkpoint — with a fake text encoder whose output we can compute by hand
def _fake_encoder(texts):
    return F.normalize(torch.tensor([[len(t), t.count("a"), 1.0] for t in texts]), dim=-1)
_W = build_zeroshot_weights(_fake_encoder, ["cat", "zebra"], ["{}", "a {}!"])
_exp = F.normalize(_fake_encoder(["zebra", "a zebra!"]).mean(0), dim=-1)
assert _W.shape == (2, 3), f"expected (2, 3), got {tuple(_W.shape)}"
assert torch.allclose(_W.norm(dim=-1), torch.ones(2), atol=1e-5), "rows must be unit-norm"
assert torch.allclose(_W[1], _exp, atol=1e-5), "average the template embeddings, then re-normalize"
print("✔️ zero-shot weights ok")

# Apply it to the toy model. Our text tower is a bag of words, so word order in templates doesn't matter.
TOY_CLASSES = [f"{c} {w}" for w in DIGIT_WORDS for c in COLOR_NAMES]         # 30 concepts: "red zero", ...
TOY_TEMPLATES = ["a {} handwritten", "{}", "a sloppy {}", "a {} number"]

@torch.no_grad()
def toy_zero_shot(model, templates):
    enc = lambda texts: model.encode_text(tokenize(texts).to(DEVICE))
    W = build_zeroshot_weights(enc, TOY_CLASSES, templates)
    g = torch.Generator().manual_seed(123)
    col = torch.randint(0, 3, (len(Xte),), generator=g)
    U = torch.cat([model.encode_image(colorize(Xte[i:i + 1000], col[i:i + 1000]).to(DEVICE)) for i in range(0, len(Xte), 1000)])
    pred = (U @ W.T).argmax(1).cpu()
    pd, pc = pred // 3, pred % 3
    return {"digit": (pd == Yte).float().mean().item(), "color": (pc == col).float().mean().item(),
            "both": ((pd == Yte) & (pc == col)).float().mean().item()}

for name, tmpls in [("single prompt", ["{}"]), ("ensemble of 4", TOY_TEMPLATES)]:
    r = toy_zero_shot(toy_clip, tmpls)
    print(f"toy zero-shot ({name:13s}) | digit acc {r['digit']:.3f} | color acc {r['color']:.3f} | both {r['both']:.3f}   (chance: 0.1 / 0.33 / 0.033)")

> **🍯 Key takeaway:** Paired data alone teaches alignment, even that "seven" and "7" mean the same thing. Duplicate concepts within a batch act as *false negatives* and set a floor on the loss.

## Part 3 — The real thing: pretrained CLIP for zero-shot CIFAR-10 🖼️

OpenAI's CLIP (Radford et al., 2021) trained exactly this loss on 400M web image–text pairs with a ViT image tower and a Transformer text tower. We'll use `openai/clip-vit-base-patch32` (~600 MB download, cached after the first run) on a few hundred **CIFAR-10** test images — classes it never saw labels for.

Every download is wrapped in `try/except`: if the model can't be fetched, the cells below print a message and skip instead of crashing.

In [ ]:
CLIP_NAME = "openai/clip-vit-base-patch32"
CLIP_OK = False
try:
    from transformers import CLIPModel, CLIPProcessor
    from transformers.utils import logging as hf_logging
    import huggingface_hub
    hf_logging.set_verbosity_error(); huggingface_hub.utils.logging.set_verbosity_error()   # keep the output tidy
    clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE).eval()
    clip_proc = CLIPProcessor.from_pretrained(CLIP_NAME)
    CLIP_OK = True
    print(f"✅ loaded {CLIP_NAME}: {sum(p.numel() for p in clip_model.parameters()) / 1e6:.0f}M params, "
          f"embedding dim {clip_model.config.projection_dim}, learned 1/τ = {clip_model.logit_scale.exp().item():.1f}")
except Exception as e:
    print(f"⚠️ Could not load {CLIP_NAME} ({type(e).__name__}: {e}).\n   Pretrained-CLIP cells will be skipped; the from-scratch parts still work.")

CIFAR_CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
N_POOL = 40 if FAST else (1000 if DEVICE == "cuda" else 300)
pool_imgs, pool_labels = [], None
try:
    print("fetching the CIFAR-10 test set...")
    try:   # fast path: the Hugging Face Hub mirror (~25 MB, usually seconds)
        from datasets import load_dataset
        with quiet():
            hf_cifar = load_dataset("uoft-cs/cifar10", split="test")
        sel = np.random.RandomState(0).choice(len(hf_cifar), N_POOL, replace=False)
        pool_imgs = [hf_cifar[int(i)]["img"].convert("RGB") for i in sel]
        pool_labels = np.array([hf_cifar[int(i)]["label"] for i in sel])
    except Exception:   # fallback: torchvision's original download (~170 MB, can be slow)
        with quiet():
            cifar = datasets.CIFAR10(DATA_ROOT, train=False, download=True)
        sel = np.random.RandomState(0).choice(len(cifar), N_POOL, replace=False)
        pool_imgs = [cifar[int(i)][0] for i in sel]
        pool_labels = np.array([cifar[int(i)][1] for i in sel])
    print(f"✅ CIFAR-10 pool: {len(pool_imgs)} test images")
except Exception as e:
    print(f"⚠️ CIFAR-10 download failed ({type(e).__name__}); using sklearn's two sample photos (no labels)")
    from sklearn.datasets import load_sample_images
    from PIL import Image
    pool_imgs = [Image.fromarray(im) for im in load_sample_images().images]

def _as_emb(out):
    """transformers 4.x returns a tensor here; 5.x returns a ModelOutput whose .pooler_output is the projected embedding."""
    return out if torch.is_tensor(out) else out.pooler_output

@torch.no_grad()
def clip_encode_images(images, bs=64):
    out = []
    for i in range(0, len(images), bs):
        px = clip_proc(images=images[i:i + bs], return_tensors="pt")["pixel_values"].to(DEVICE)
        out.append(F.normalize(_as_emb(clip_model.get_image_features(pixel_values=px)), dim=-1).cpu())
    return torch.cat(out)

@torch.no_grad()
def clip_encode_text(texts):
    inp = clip_proc(text=texts, return_tensors="pt", padding=True).to(DEVICE)
    return F.normalize(_as_emb(clip_model.get_text_features(**inp)), dim=-1).cpu()

if CLIP_OK:
    t0 = time.time(); IMG_EMB = clip_encode_images(pool_imgs)
    print(f"encoded {len(pool_imgs)} images -> {tuple(IMG_EMB.shape)} in {time.time() - t0:.1f}s")

### Prompt engineering & prompt ensembling

A bare label like `"cat"` is an unusual caption — web alt-text looks more like *"a photo of a cat."* Matching the **training caption distribution** helps; averaging over several phrasings (**ensembling**) helps more. The CLIP paper reports gains from both (on ImageNet, ensembling 80 templates added several points of accuracy).

🔮 **Predict-then-run:** rank the three strategies — bare label, `"a photo of a {}."`, and an ensemble of 10 templates — by accuracy.

In [ ]:
ENSEMBLE = ["a photo of a {}.", "a blurry photo of a {}.", "a black and white photo of a {}.", "a low contrast photo of a {}.",
            "a high contrast photo of a {}.", "a bad photo of a {}.", "a good photo of a {}.", "a photo of a small {}.",
            "a photo of a big {}.", "a pixelated photo of a {}."]
STRATEGIES = {"bare label": ["{}"], "'a photo of a {}.'": ["a photo of a {}."], "ensemble (10)": ENSEMBLE}

if CLIP_OK and pool_labels is not None:
    accs, preds = {}, {}
    for name, tmpls in STRATEGIES.items():
        W = build_zeroshot_weights(clip_encode_text, CIFAR_CLASSES, tmpls)
        preds[name] = (IMG_EMB @ W.T).argmax(1).numpy()
        accs[name] = (preds[name] == pool_labels).mean()
        print(f"{name:22s} zero-shot accuracy: {accs[name]:.3f}")
    W_ENS = build_zeroshot_weights(clip_encode_text, CIFAR_CLASSES, ENSEMBLE)

    from sklearn.metrics import confusion_matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(list(accs), list(accs.values()), color=[HIVE[4], HIVE[3], HIVE[0]])
    axes[0].set_ylim(min(accs.values()) - 0.1, 1.0); axes[0].set_ylabel("accuracy"); axes[0].set_title(f"Zero-shot CIFAR-10 ({len(pool_labels)} images)")
    cm = confusion_matrix(pool_labels, preds["ensemble (10)"], labels=range(10))
    axes[1].imshow(cm, cmap="YlOrBr"); axes[1].set_xticks(range(10)); axes[1].set_xticklabels(CIFAR_CLASSES, rotation=45, ha="right")
    axes[1].set_yticks(range(10)); axes[1].set_yticklabels(CIFAR_CLASSES); axes[1].set_xlabel("predicted"); axes[1].set_ylabel("true")
    for i in range(10):
        for j in range(10):
            if cm[i, j]: axes[1].text(j, i, cm[i, j], ha="center", va="center", fontsize=7)
    axes[1].set_title("Confusion (ensemble)"); plt.tight_layout(); plt.show()

    probs = (clip_model.logit_scale.exp().item() * IMG_EMB[:8] @ W_ENS.T).softmax(-1)
    fig, axes = plt.subplots(2, 8, figsize=(18, 4.6), gridspec_kw={"height_ratios": [1.3, 1]})
    for i in range(8):
        axes[0, i].imshow(pool_imgs[i]); axes[0, i].axis("off"); axes[0, i].set_title(f"true: {CIFAR_CLASSES[pool_labels[i]]}", fontsize=9)
        top = probs[i].topk(3)
        axes[1, i].barh(range(3)[::-1], top.values.numpy(), color=[HIVE[0] if k == pool_labels[i] else HIVE[1] for k in top.indices.tolist()])
        axes[1, i].set_yticks(range(3)[::-1]); axes[1, i].set_yticklabels([CIFAR_CLASSES[k] for k in top.indices]); axes[1, i].set_xlim(0, 1)
    plt.suptitle("Top-3 zero-shot predictions (gold = correct)"); plt.tight_layout(); plt.show()
else:
    print("⏭️ skipped (needs pretrained CLIP and labelled CIFAR-10)")

**Reading the numbers.** On a few hundred images, the standard error of an accuracy near 0.9 is about ±2 points, so small differences between prompt strategies are within noise here (the bare label usually trails). The CLIP paper measured prompt-engineering and ensembling gains on full test sets across many datasets — rerun with a bigger `N_POOL` on a GPU to see them more clearly.

### Retrieval: text → image and image → text 🔎

The same similarity matrix runs **search** in both directions. Free-form queries work — no need to stick to the 10 class names.

In [ ]:
QUERIES = ["a red car", "an animal with antlers", "a ship on the water", "a cute puppy", "an airplane in a blue sky", "a frog"]
if CLIP_OK:
    Q = clip_encode_text(QUERIES)
    S = Q @ IMG_EMB.T                                              # (queries, images)
    k = min(5, len(pool_imgs))
    fig, axes = plt.subplots(len(QUERIES), k, figsize=(1.9 * k, 2.0 * len(QUERIES)), squeeze=False)
    for r, q in enumerate(QUERIES):
        for c, j in enumerate(S[r].topk(k).indices.tolist()):
            axes[r, c].imshow(pool_imgs[j]); axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
            axes[r, c].set_xlabel(f"{S[r, j]:.2f}", fontsize=8)
        axes[r, 0].set_ylabel(q, fontsize=9, rotation=0, ha="right", va="center")
    plt.suptitle("Text → image retrieval (cosine similarity under each)"); plt.tight_layout(); plt.show()

    CAPTIONS = ["a photo of a cat", "a photo of a dog", "a photo of a truck", "a photo of a horse", "a photo of a bird",
                "a photo of an airplane", "a photo of a boat", "a blurry photo", "a photo taken at night", "a photo of food"]
    C = clip_encode_text(CAPTIONS)
    n_show = min(4, len(pool_imgs))
    fig, axes = plt.subplots(1, n_show, figsize=(4.2 * n_show, 3.2), squeeze=False)
    for i in range(n_show):
        sims = (IMG_EMB[i] @ C.T); top = sims.topk(3)
        axes[0, i].imshow(pool_imgs[i]); axes[0, i].axis("off")
        axes[0, i].set_title("\n".join(f"{CAPTIONS[j]} ({s:.2f})" for s, j in zip(top.values.tolist(), top.indices.tolist())), fontsize=8.5)
    plt.suptitle("Image → text retrieval: top-3 captions", y=1.08); plt.show()
else:
    print("⏭️ skipped (needs pretrained CLIP)")

### 🧪 Your Turn #3 — Recall@K

Retrieval quality is usually reported as **Recall@K**: the fraction of queries whose correct item appears among the top-$K$ results. Implement `recall_at_k(sim, gt, ks)` where `sim` is `(n_queries, n_items)`, `gt[i]` is the index of the correct item for query `i`, and the function returns a dict `{k: recall}`.

*Hint:* `sim.topk(k, dim=1).indices` and compare to `gt[:, None]`.

In [ ]:
def recall_at_k(sim, gt, ks=(1, 5)):
    # TODO
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def recall_at_k(sim, gt, ks=(1, 5)):
    gt = torch.as_tensor(gt)
    out = {}
    for k in ks:
        top = sim.topk(min(k, sim.shape[1]), dim=1).indices
        out[k] = (top == gt[:, None]).any(1).float().mean().item()
    return out

In [ ]:
# ✔️ Checkpoint
_sim = torch.tensor([[0.9, 0.1, 0.0], [0.2, 0.3, 0.5], [0.1, 0.8, 0.7]])
_r = recall_at_k(_sim, torch.tensor([0, 1, 2]), ks=(1, 2, 3))
assert abs(_r[1] - 1 / 3) < 1e-6 and abs(_r[2] - 1.0) < 1e-6 and abs(_r[3] - 1.0) < 1e-6, _r
print("✔️ recall_at_k ok:", _r)

if CLIP_OK and pool_labels is not None:
    print("CLIP image→class  Recall@K:", {k: round(v, 3) for k, v in recall_at_k(IMG_EMB @ W_ENS.T, pool_labels, ks=(1, 3, 5)).items()})
# Toy model: retrieve the right caption among 500 test pairs (only counts as correct if it's *that* pair's caption)
with torch.no_grad():
    _img, _tok, _y, _c, _ = get_batch(Xte, Yte, 500, torch.Generator().manual_seed(7))
    _S = toy_clip.encode_image(_img.to(DEVICE)) @ toy_clip.encode_text(_tok.to(DEVICE)).T
print("toy  image→caption Recall@K (500 pairs, exact pair only):", {k: round(v, 3) for k, v in recall_at_k(_S.cpu(), torch.arange(500), ks=(1, 5, 20)).items()},
      "\n  ↳ low R@1 is expected: ~17 captions describe the same concept, the same false-negative issue as in training!")

### Looking inside the joint embedding space 🔭

Let's project CLIP's 512-d embeddings to 2D. Two things to look for:
1. **Semantic clusters** — images of the same class should cluster (t-SNE is good at showing this).
2. **The modality gap** (Liang et al., 2022) — in a *joint* PCA of images *and* texts, the two modalities often occupy **separate regions** of the sphere even though matching pairs are "closest". Contrastive loss only cares about *relative* similarities, so a constant offset between modalities is never penalized.

### 🧪 Your Turn #4 — measure the modality gap
`modality_gap(img_emb, txt_emb)`: the Euclidean distance between the **mean** image embedding and the **mean** text embedding (inputs are already normalized).

In [ ]:
def modality_gap(img_emb, txt_emb):
    # TODO
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def modality_gap(img_emb, txt_emb):
    return (img_emb.mean(0) - txt_emb.mean(0)).norm().item()

In [ ]:
# ✔️ Checkpoint
_a = F.normalize(torch.randn(500, 32), dim=-1)
assert modality_gap(_a, _a) < 1e-6, "identical sets have zero gap"
assert abs(modality_gap(torch.tensor([[1.0, 0.0]]), torch.tensor([[0.0, 1.0]])) - math.sqrt(2)) < 1e-6
print("✔️ modality_gap ok")

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
if CLIP_OK and pool_labels is not None:
    TXT_EMB = clip_encode_text([f"a photo of a {c}." for c in CIFAR_CLASSES])
    print(f"CLIP modality gap: {modality_gap(IMG_EMB, TXT_EMB):.3f}   (for scale: a random unit-vector pair is ~1.41 apart)")
    both = torch.cat([IMG_EMB, TXT_EMB]).numpy()
    P = PCA(2).fit_transform(both)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    cmap = plt.get_cmap("tab10")
    axes[0].scatter(P[:len(IMG_EMB), 0], P[:len(IMG_EMB), 1], c=[cmap(l) for l in pool_labels], s=12, alpha=0.7)
    axes[0].scatter(P[len(IMG_EMB):, 0], P[len(IMG_EMB):, 1], c=[cmap(i) for i in range(10)], s=260, marker="*", edgecolor="k")
    for i, c in enumerate(CIFAR_CLASSES):
        axes[0].annotate(c, P[len(IMG_EMB) + i], fontsize=8, xytext=(4, 4), textcoords="offset points")
    axes[0].set_title("Joint PCA: dots = images, ★ = text prompts\n(notice the two modalities sit apart: the modality gap)")
    Z = TSNE(2, perplexity=min(30, len(IMG_EMB) // 4), init="pca", random_state=0).fit_transform(IMG_EMB.numpy())
    for l in range(10):
        m = pool_labels == l
        axes[1].scatter(Z[m, 0], Z[m, 1], s=12, color=cmap(l), label=CIFAR_CLASSES[l])
    axes[1].legend(fontsize=8, markerscale=2, ncol=2); axes[1].set_title("t-SNE of CLIP image embeddings")
    for ax in axes: ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()
else:
    print("⏭️ CLIP visualization skipped")

# The same picture for our toy model (30 concepts)
with torch.no_grad():
    _img, _tok, _y, _c, _ = get_batch(Xte, Yte, 600, torch.Generator().manual_seed(3))
    tU = toy_clip.encode_image(_img.to(DEVICE)).cpu()
    tV = toy_clip.encode_text(tokenize([f"a {c} handwritten {w}" for w in DIGIT_WORDS for c in COLOR_NAMES]).to(DEVICE)).cpu()
print(f"toy model modality gap: {modality_gap(tU, tV):.3f}")
P = PCA(2).fit_transform(torch.cat([tU, tV]).numpy())
plt.figure(figsize=(6.5, 5.5))
plt.scatter(P[:600, 0], P[:600, 1], c=[COLOR_RGB[c].numpy() for c in _c], s=10, alpha=0.6)
plt.scatter(P[600:, 0], P[600:, 1], c=[COLOR_RGB[i % 3].numpy() for i in range(30)], marker="*", s=200, edgecolor="k")
plt.title("Toy model joint PCA (color = ink color, ★ = text)"); plt.xticks([]); plt.yticks([]); plt.show()

> **🍯 Key takeaway:** Zero-shot classification is nearest-prompt search: phrasing and ensembling prompts matter, and the same similarity matrix powers retrieval in both directions. Images and texts still live in visibly separate regions (the modality gap).

## Part 4 — SigLIP: replace the softmax with sigmoids 🧮

**Problem with softmax InfoNCE:** every row's normalizer $\sum_j e^{\ell_{ij}}$ couples *all* pairs in the batch. In distributed training you must all-gather every embedding to compute it, and the loss depends on batch size in subtle ways.

**SigLIP (Zhai et al., 2023)** treats every (image, text) cell of the $N\times N$ matrix as an **independent binary classification**: positive on the diagonal, negative elsewhere.
$$\mathcal{L}_{\text{SigLIP}} = -\frac{1}{N}\sum_{i=1}^N\sum_{j=1}^N \log \sigma\big(z_{ij}\,(t\,u_i^\top v_j + b)\big),\qquad z_{ij} = \begin{cases}+1 & i=j\\ -1 & i\ne j\end{cases}$$
with learnable temperature $t = e^{t'}$ and bias $b$, initialized to $t' = \log 10$ and $b = -10$. The negative bias matters: there are $N-1$ negatives per positive, so starting with "everything is probably a negative" avoids huge initial gradients. No global normalization → simpler, memory-efficient chunked implementations, and strong results especially at small and moderate batch sizes. SigLIP encoders (and SigLIP 2, Tschannen et al., 2025) are widely used as VLM vision towers.

### 🧪 Your Turn #5 — implement the SigLIP loss
`siglip_loss(img_emb, txt_emb, t, b)`: normalize, `logits = t * img @ txt.T + b`, labels $z = 2I - 1$, return $-\frac{1}{N}\sum_{ij}\log\sigma(z_{ij}\,\text{logits}_{ij})$. *Hint:* `F.logsigmoid` is numerically stable.

In [ ]:
def siglip_loss(img_emb, txt_emb, t, b):
    # TODO
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def siglip_loss(img_emb, txt_emb, t, b):
    img, txt = F.normalize(img_emb, dim=-1), F.normalize(txt_emb, dim=-1)
    logits = t * img @ txt.T + b
    z = 2 * torch.eye(len(img), device=img.device) - 1
    return -F.logsigmoid(z * logits).sum() / len(img)

In [ ]:
# ✔️ Checkpoint — explicit loop with the textbook sigmoid
def _reference_siglip(a, bb, t, b):
    a = F.normalize(a, dim=-1); bb = F.normalize(bb, dim=-1); N = len(a); tot = 0.0
    for i in range(N):
        for j in range(N):
            z = 1.0 if i == j else -1.0
            tot += -math.log(1 / (1 + math.exp(-z * (t * float(a[i] @ bb[j]) + b))))
    return tot / N
torch.manual_seed(0)
_a, _b = torch.randn(12, 8), torch.randn(12, 8)
assert abs(siglip_loss(_a, _b, 10.0, -10.0).item() - _reference_siglip(_a, _b, 10.0, -10.0)) < 1e-4
assert abs(siglip_loss(_a, _b, 2.0, 0.5).item() - _reference_siglip(_a, _b, 2.0, 0.5)) < 1e-4
print("✔️ siglip_loss matches the reference")

seed_everything(0)
toy_siglip = TinyCLIP(len(VOCAB), init_scale=10.0, init_bias=-10.0).to(DEVICE)
toy_siglip, hist_sig = train_contrastive(toy_siglip, "siglip")
r_clip, r_sig = toy_zero_shot(toy_clip, TOY_TEMPLATES), toy_zero_shot(toy_siglip, TOY_TEMPLATES)
print(f"zero-shot 'both' accuracy  CLIP loss: {r_clip['both']:.3f}  |  SigLIP loss: {r_sig['both']:.3f}")
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_sim(toy_clip, "toy model, CLIP (softmax) loss", axes[0]); plot_sim(toy_siglip, f"toy model, SigLIP loss (b = {toy_siglip.bias.item():.1f})", axes[1])
plt.tight_layout(); plt.show()

> **🍯 Key takeaway:** SigLIP replaces the batch-coupled softmax with independent per-pair sigmoids plus a learned bias. That is simpler to shard across devices and strong at modest batch sizes.

## Part 5 — From CLIP to vision-language models (VLMs) 🧠👁️

A contrastive encoder gives one vector per image — great for search and classification, but an LLM needs **a sequence of tokens** to reason about. Modern VLMs connect the two in a few standard ways:

| Design | How vision enters the LLM | Examples |
|---|---|---|
| **Projector ("LLaVA-style")** | Take the vision encoder's **patch tokens**, map them with a linear layer or small MLP into the LLM's embedding space, and simply **prepend** them to the text tokens. | LLaVA (Liu et al., 2023: CLIP ViT-L/14 + MLP projector in LLaVA-1.5), PaliGemma (Beyer et al., 2024: SigLIP + Gemma) |
| **Learned queries / resampler** | A small transformer with a fixed set of learned query tokens cross-attends to the patch features and outputs a **fixed number** of visual tokens. | BLIP-2's Q-Former (Li et al., 2023), Flamingo's Perceiver Resampler (Alayrac et al., 2022) |
| **Cross-attention layers** | Keep the LLM's sequence text-only; insert (gated) **cross-attention** layers that read the visual features. | Flamingo (Alayrac et al., 2022) |
| **Natively multimodal** | Train one model on interleaved image/text (and audio/video) tokens from early on. | Many frontier assistant models (details often unpublished) |

Typical training: (1) **align** — freeze the vision encoder and LLM, train only the projector on image–caption pairs; (2) **instruction-tune** on visual question-answer / chat data, often unfreezing the LLM. Practical details that matter a lot: image resolution and tiling (more patches = more tokens = more cost), and which encoder layer you tap (LLaVA uses the penultimate layer).

Below is the **shape-level plumbing** of the projector design using CLIP's real vision tower.

In [ ]:
D_LLM = 2048                                   # hidden size of a hypothetical small LLM
if CLIP_OK:
    with torch.no_grad():
        px = clip_proc(images=pool_imgs[:2], return_tensors="pt")["pixel_values"].to(DEVICE)
        vis = clip_model.vision_model(pixel_values=px, output_hidden_states=True)
        patch_feats = vis.hidden_states[-2][:, 1:, :].cpu()   # penultimate layer, drop the [CLS] token
else:
    patch_feats = torch.randn(2, 49, 768)          # same shape CLIP ViT-B/32 would give at 224px
print("patch features from the vision tower:", tuple(patch_feats.shape), " ← (batch, 7x7 patches, 768)")

projector = nn.Sequential(nn.Linear(patch_feats.shape[-1], D_LLM), nn.GELU(), nn.Linear(D_LLM, D_LLM))   # LLaVA-1.5-style 2-layer MLP
visual_tokens = projector(patch_feats)
text_token_embs = torch.randn(2, 12, D_LLM)    # stand-in for the LLM's embeddings of "Describe this image in one sentence."
llm_input = torch.cat([visual_tokens, text_token_embs], dim=1)
print("visual tokens:", tuple(visual_tokens.shape), "| text tokens:", tuple(text_token_embs.shape), "| LLM input sequence:", tuple(llm_input.shape))
print(f"projector params: {sum(p.numel() for p in projector.parameters()) / 1e6:.1f}M  (often the only part trained in the alignment stage)")
for name, (res, patch) in {"ViT-B/32 @224": (224, 32), "ViT-L/14 @336 (LLaVA-1.5)": (336, 14), "ViT @448, patch 14": (448, 14)}.items():
    print(f"  {name:28s} -> {(res // patch) ** 2:5d} visual tokens per image")

> **🍯 Key takeaway:** VLMs turn a vision encoder's patch tokens into "words" an LLM can read, via a projector, a resampler or cross-attention. The number of visual tokens is a key cost knob.

## 🏆 Boss Level

1. **Fix the false negatives.** In the toy setup we *know* which pairs share a concept. Replace the one-hot targets of the CLIP loss with soft targets spread uniformly over all same-concept items in the batch (cf. Supervised Contrastive Learning, Khosla et al., 2020). Does the loss floor disappear? Does exact-pair Recall@1 change, and does that even make sense to measure now?
2. **Compositional generalization.** Remove every *"blue seven"* from training (filter `get_batch`), retrain, and test zero-shot on blue sevens. Does the model compose "blue" + "seven" it has seen separately? Swap the bag-of-words text tower for a 2-layer Transformer and compare.
3. **Zero-shot vs. few-shot.** Fit `sklearn.linear_model.LogisticRegression` on CLIP image features with 1, 4 and 16 labelled examples per CIFAR class. How many shots does a linear probe need to beat zero-shot? (Radford et al. found zero-shot CLIP matched a 4-shot linear probe on average across datasets.) Try renaming classes (e.g. `"automobile"` → `"car"`) — prompt engineering of class names is real!

## 🍯 Key Takeaways

- **Contrastive learning = classification without labels:** inside each batch, every image must pick its own caption from $N$ candidates, and every caption its own image.
- **The CLIP loss** is symmetric cross-entropy over the cosine-similarity matrix with a learnable temperature; a random model scores about $\log N$.
- **False negatives are real:** duplicate concepts in a batch cap how low the loss can go. Bigger, more diverse data and smarter targets help.
- **Zero-shot = nearest text prompt.** Prompt templates and ensembling are free accuracy, and Recall@K measures retrieval.
- **Joint spaces have structure:** semantic clusters, plus a **modality gap** between image and text embeddings.
- **SigLIP** swaps the softmax for independent sigmoids with a learned bias, which makes it simpler to scale.
- **VLMs** = vision encoder (often CLIP/SigLIP) → projector/resampler → LLM tokens. Image resolution decides how many tokens you pay for.

## 🎤 Explain it to a friend

Explain to a friend how a computer can sort photos into categories it was **never trained on**, like "a photo of a frog", just by having read a lot of image captions on the internet. Use the analogy of a party game: everyone in the room holds either a photo or a caption, and the goal is to find your partner. What does the computer learn by playing that game millions of times?

## 💼 Interview corner

**Q1. Why does CLIP training benefit from very large batches, and how does SigLIP change the picture?**
<details><summary>Answer</summary>

Each example's negatives are the other items in the batch. More negatives give a harder, more informative task and a tighter InfoNCE bound on mutual information ($I \ge \log N - \mathcal{L}$). CLIP used a batch of 32,768. But the softmax normalizer couples every pair in the batch, so the full similarity matrix (or all embeddings) must be gathered across devices. SigLIP treats each pair as an independent binary classification with a learned bias. No global normalization is needed, so it can be computed in chunks with less memory and communication, and Zhai et al. found it performs better than softmax at smaller batch sizes, with the gap shrinking as the batch grows.
</details>

**Q2. How does zero-shot classification with CLIP work, and how can you improve its accuracy without any training?**
<details><summary>Answer</summary>

Encode one text prompt per class (e.g. "a photo of a {class}."), L2-normalize, and classify each image by the highest cosine similarity; the text embeddings act as the weights of a linear classifier. Training-free improvements: prompt templates that match caption style, **prompt ensembling** (average the normalized embeddings of many templates per class), better class names (synonyms, disambiguation such as "crane (bird)"), and domain-specific context ("a satellite photo of …"). Beyond zero-shot, a cheap linear probe on the frozen features often helps with only a few labelled examples.
</details>

**Q3. What is the "modality gap", and does it matter?**
<details><summary>Answer</summary>

In contrastively trained models, image and text embeddings occupy separate regions (cones) of the hypersphere, so the centroids of the two modalities are far apart even though matching pairs are relatively closest (Liang et al., 2022). The contrastive loss depends only on relative similarities, so it never removes the constant offset, and initialization and temperature shape its size. It matters when you mix modalities directly, e.g. using text embeddings as stand-ins for image embeddings, or thresholding absolute similarity scores. Centering each modality, or learning a small adapter, is a common fix.
</details>

## 🐝 Fun fact

CLIP's abstract reports matching the accuracy of the original ResNet-50 on ImageNet **zero-shot**, without using any of ImageNet's 1.28 million labelled training images. It learned the categories from the way people caption pictures on the web.

## 📚 Further reading

- Radford et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision.* ICML. (CLIP)
- Jia et al. (2021). *Scaling Up Visual and Vision-Language Representation Learning With Noisy Text Supervision.* ICML. (ALIGN)
- van den Oord, Li & Vinyals (2018). *Representation Learning with Contrastive Predictive Coding.* arXiv:1807.03748. (InfoNCE)
- Chen, Kornblith, Norouzi & Hinton (2020). *A Simple Framework for Contrastive Learning of Visual Representations.* ICML. (SimCLR)
- Zhai, Mustafa, Kolesnikov & Beyer (2023). *Sigmoid Loss for Language Image Pre-Training.* ICCV. (SigLIP)
- Tschannen et al. (2025). *SigLIP 2: Multilingual Vision-Language Encoders with Improved Semantic Understanding, Localization, and Dense Features.* arXiv preprint.
- Liang, Zhang, Kwon, Yeung & Zou (2022). *Mind the Gap: Understanding the Modality Gap in Multi-modal Contrastive Representation Learning.* NeurIPS.
- Cherti et al. (2023). *Reproducible Scaling Laws for Contrastive Language-Image Learning.* CVPR. (OpenCLIP)
- Khosla et al. (2020). *Supervised Contrastive Learning.* NeurIPS.
- Alayrac et al. (2022). *Flamingo: a Visual Language Model for Few-Shot Learning.* NeurIPS.
- Li, Li, Savarese & Hoi (2023). *BLIP-2: Bootstrapping Language-Image Pre-training with Frozen Image Encoders and Large Language Models.* ICML.
- Liu, Li, Wu & Lee (2023). *Visual Instruction Tuning.* NeurIPS. (LLaVA)
- Beyer et al. (2024). *PaliGemma: A versatile 3B VLM for transfer.* arXiv preprint.

🐝 *Buzz says: you've taught pixels and words to share one hive mind. That's a wrap on Generative Worlds!*